In [6]:
import requests # для отправки запросов в интернет
import pandas as pd # для работы с таблицами
import time # для пауз, чтобы сервер не забанил нас за слишком частые запросы
from google.colab import userdata

ACCESS_TOKEN = userdata.get('VK_TOKEN')
API_VERSION = "5.199"

# Подборка открытых сообществ (официальный паблик авторов + крупные обзорщики маркетплейсов и лайфстайл)
communities = [
    "vk_authors",        # Официальное сообщество авторов ВК (из брифа)
    "ozon",              # Официальный Ozon (для сравнения товарных форматов)
    "wildberries_ru",    # Крупный e-com референс
    "nahodki_wb_ozon",   # Типичный паблик с подборками товаров
    "aliexpress",        # Крупный e-com с геймификацией
    "vogue_russia",      # Лайфстайл / медиа формат
    "thevillage",        # Городское медиа / нативные интеграции
    "lifehacker_ru"      # Контентный паблик с партнерскими ссылками
]

def get_posts(domain, count=100):
    base = "https://" + "api.vk.com"
    method = "/method/wall.get"
    url = base + method
    params = {
        "domain": domain,
        "count": count,
        "access_token": ACCESS_TOKEN,
        "v": API_VERSION
    }
    response = requests.get(url, params=params).json()

    if "error" in response:
        print(f"⚠️ Ошибка для {domain}: {response['error']['error_msg']}")
        return []

    return response["response"]["items"]

all_data = []

for domain in communities:
    posts = get_posts(domain, count=100)
    print(f"✅ {domain}: загружено {len(posts)} постов")
    for post in posts:
        attachments = post.get("attachments", [])
        att_types = [att["type"] for att in attachments]
        text = post.get("text", "").lower()

        # Определяем признаки интерактивности (опросы или призывы в тексте)
        is_interactive = ("poll" in att_types) or any(
            word in text for word in ["опрос", "выбирай", "голосуй", "пиши в комментариях", "угадай", "конкурс", "розыгрыш", "баттл"]
        )
        # Определяем признаки товарного/партнерского поста (ссылка, товар, маркетплейс)
        is_shop_post = ("link" in att_types) or ("market" in att_types) or any(
            word in text for word in ["ozon", "озон", "артикул", "купить", "заказать", "скидк", "промокод", "шопс"]
        )

        all_data.append({
            "community": domain,
            "post_id": post["id"],
            "date": pd.to_datetime(post["date"], unit="s"),
            "text_length": len(text),
            "is_interactive": is_interactive,
            "is_shop_post": is_shop_post,
            "has_poll": "poll" in att_types,
            "has_link": "link" in att_types,
            "has_video": "video" in att_types,
            "attachments": ", ".join(att_types) if att_types else "text_only",
            "views": post.get("views", {}).get("count", 0),
            "likes": post.get("likes", {}).get("count", 0),
            "comments": post.get("comments", {}).get("count", 0),
            "reposts": post.get("reposts", {}).get("count", 0)
        })
    time.sleep(0.4)

df = pd.DataFrame(all_data)
df.to_csv("vk_posts_dataset.csv", index=False)
print(f"\n🎉 Готово! Всего собрано постов: {len(df)}")
df.head()


⚠️ Ошибка для vk_authors: One of the parameters specified was missing or invalid: domain not domain
✅ vk_authors: загружено 0 постов
✅ ozon: загружено 100 постов
✅ wildberries_ru: загружено 6 постов
✅ nahodki_wb_ozon: загружено 100 постов
✅ aliexpress: загружено 100 постов
✅ vogue_russia: загружено 0 постов
✅ thevillage: загружено 100 постов
✅ lifehacker_ru: загружено 100 постов

🎉 Готово! Всего собрано постов: 506


,community,post_id,date,text_length,is_interactive,is_shop_post,has_poll,has_link,has_video,attachments,views,likes,comments,reposts
0,ozon,2574722,2026-11-23 14:00:02,281,False,True,False,False,False,"photo, photo, photo, photo, photo, photo",26288,219,77,12
1,ozon,2574596,2026-11-23 11:00:02,305,False,True,False,False,True,"photo, video",30854,317,154,6
2,ozon,2574525,2026-11-23 09:15:00,831,True,False,False,False,False,photo,61350,494,67,28
3,ozon,2574345,2026-11-23 07:00:01,424,False,True,False,False,False,"photo, photo, photo, photo, photo, photo, phot...",24077,234,25,3
4,ozon,2573411,2026-11-22 09:00:00,0,False,False,False,False,False,text_only,29139,314,86,7


In [7]:
import pandas as pd

df = pd.read_csv("vk_posts_dataset.csv")
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   community       506 non-null    object
 1   post_id         506 non-null    int64 
 2   date            506 non-null    object
 3   text_length     506 non-null    int64 
 4   is_interactive  506 non-null    bool  
 5   is_shop_post    506 non-null    bool  
 6   has_poll        506 non-null    bool  
 7   has_link        506 non-null    bool  
 8   has_video       506 non-null    bool  
 9   attachments     506 non-null    object
 10  views           506 non-null    int64 
 11  likes           506 non-null    int64 
 12  comments        506 non-null    int64 
 13  reposts         506 non-null    int64 
dtypes: bool(5), int64(6), object(3)
memory usage: 38.2+ KB


,0
community,0
post_id,0
date,0
text_length,0
is_interactive,0
is_shop_post,0
has_poll,0
has_link,0
has_video,0
attachments,0


In [10]:
import pandas as pd

df = pd.read_csv("vk_posts_dataset.csv")
df = df[df["views"] > 0].copy()
df["er_percent"] = (df["likes"] + df["comments"] + df["reposts"]) / df["views"] * 100 #расчёт воволечённости
df.groupby("community").agg(total_posts=("post_id", "count"), interactive_share=("is_interactive", "mean"))
df["is_interactive"].mean() * 100
df.groupby("is_interactive")["er_percent"].agg(["count", "mean", "median"])

,count,mean,median
is_interactive,,,
False,447,4.973495,0.386847
True,53,2.215694,0.311656
